# Content-Yield Gate — Full Range (headless GPU) — v3

v3: **float32** (half the GPU memory; fixes the likely T4 OOM) + frees GPU memory
between roots. Steps: Import → GPU T4 + Internet On → **Save & Run All (Commit)** →
close tab → return → copy the last cell's `yield_report.json`.


In [ ]:
!nvidia-smi -L || echo 'NO GPU — Settings -> Accelerator -> GPU'

In [ ]:
import base64, zipfile, io, os
_ZIP_B64 = (
    "..."
)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(_ZIP_B64))) as z: z.extractall('/kaggle/working/poker')
print('unpacked ->', sorted(os.listdir('/kaggle/working/poker')))

In [ ]:
try:
    import cupy as cp
except Exception:
    import subprocess,sys; subprocess.run([sys.executable,'-m','pip','install','-q','cupy-cuda12x']); import cupy as cp
free,total=cp.cuda.Device(0).mem_info
nm=cp.cuda.runtime.getDeviceProperties(0)['name']; print('cupy',cp.__version__,'|',nm.decode() if isinstance(nm,bytes) else nm)
print('GPU memory: %.1f GB free / %.1f GB total'%(free/1e9,total/1e9))

In [ ]:
# Smoke: 1 root at full range, float32 (~20-35 min). Prints peak GPU memory used.
!cd /kaggle/working/poker && PYTHONPATH=src python -m pokertrainer.content_yield \
    --solver gpu --dtype float32 --n 400 --iters 300 --roots 0 --out /kaggle/working/cy_smoke
import cupy as cp; print('peak GPU mem used: %.1f GB'%(cp.get_default_memory_pool().used_bytes()/1e9))
import json; print(json.load(open('/kaggle/working/cy_smoke/yield_report.json'))['projected_accepted_per_root_full_range'],'accepted/root (full range)')

## Full run — 12 roots, float32 (~3-5 h headless). Optional: the smoke above already confirms the gate.


In [ ]:
!cd /kaggle/working/poker && PYTHONPATH=src python -m pokertrainer.content_yield \
    --solver gpu --dtype float32 --n 400 --iters 300 --out /kaggle/working/cyout

In [ ]:
import json; print(json.dumps(json.load(open('/kaggle/working/cyout/yield_report.json')), indent=1))

_If the smoke cell errors, copy its full output (or the committed version's Log) back — that shows OOM vs other errors._